In [1]:
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = PROCESSED_DIR / "splits"

TRAIN_PATH = SPLIT_DIR / "train.parquet"
TEST_PATH = SPLIT_DIR / "test.parquet"

MOVIES_PATH = PROJECT_ROOT / "data" / "raw" / "ml-20m" / "movies.csv"

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 2")

In [2]:
train_path = TRAIN_PATH.as_posix()

popularity_query = f"""
SELECT
    movieId,
    COUNT(*) AS rating_count
FROM read_parquet('{train_path}')
GROUP BY movieId
ORDER BY rating_count DESC
"""

movie_popularity = con.execute(popularity_query).df()

print(movie_popularity.shape)
movie_popularity.head(10)

(26707, 2)


,movieId,rating_count
0,296,66926
1,356,65739
2,318,62890
3,593,62805
4,480,59234
5,260,54214
6,110,53527
7,589,51702
8,2571,50918
9,527,49528


In [3]:
movies = pd.read_csv(MOVIES_PATH)

top_10 = (
    movie_popularity
    .head(10)
    .merge(movies, on="movieId", how="left")
)

top_10

,movieId,rating_count,title,genres
0,296,66926,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
1,356,65739,Forrest Gump (1994),Comedy|Drama|Romance|War
2,318,62890,"Shawshank Redemption, The (1994)",Crime|Drama
3,593,62805,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
4,480,59234,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
5,260,54214,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
6,110,53527,Braveheart (1995),Action|Drama|War
7,589,51702,Terminator 2: Judgment Day (1991),Action|Sci-Fi
8,2571,50918,"Matrix, The (1999)",Action|Sci-Fi|Thriller
9,527,49528,Schindler's List (1993),Drama|War


In [4]:
MODEL_PATH = MODEL_DIR / "popularity_baseline.parquet"

top_10.to_parquet(
    MODEL_PATH,
    index=False
)

In [5]:
test_path = TEST_PATH.as_posix()

top_10_ids = tuple(top_10["movieId"].tolist())

evaluation_query = f"""
SELECT
    COUNT(*) AS relevant_test_ratings,
    COUNT(*) FILTER (
        WHERE movieId IN {top_10_ids}
    ) AS hits
FROM read_parquet('{test_path}')
WHERE rating >= 4.0
"""

evaluation = con.execute(evaluation_query).df()

evaluation["recall_at_10"] = (
    evaluation["hits"] /
    evaluation["relevant_test_ratings"]
)

evaluation

,relevant_test_ratings,hits,recall_at_10
0,80321,3266,0.040662


In [6]:
m = 1000  # minimum ratings threshold

weighted_query = f"""
WITH global_stats AS (
    SELECT AVG(rating) AS C
    FROM read_parquet('{TRAIN_PATH.as_posix()}')
),
movie_stats AS (
    SELECT
        movieId,
        COUNT(*) AS v,
        AVG(rating) AS R
    FROM read_parquet('{TRAIN_PATH.as_posix()}')
    GROUP BY movieId
)
SELECT
    ms.movieId,
    ms.v AS rating_count,
    ROUND(ms.R, 3) AS avg_rating,
    ROUND((ms.v / (ms.v + {m})) * ms.R + ({m} / (ms.v + {m})) * gs.C, 4) AS weighted_score
FROM movie_stats ms, global_stats gs
ORDER BY weighted_score DESC
"""

weighted_popularity = con.execute(weighted_query).df()

top_10_weighted = (
    weighted_popularity
    .head(10)
    .merge(movies, on="movieId", how="left")
)

top_10_weighted.to_parquet(MODEL_DIR / "weighted_popularity_baseline.parquet", index=False)
top_10_weighted

,movieId,rating_count,avg_rating,weighted_score,title,genres
0,318,62890,4.447,4.4324,"Shawshank Redemption, The (1994)",Crime|Drama
1,858,41025,4.365,4.3449,"Godfather, The (1972)",Crime|Drama
2,50,46620,4.334,4.3174,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
3,527,49528,4.310,4.2944,Schindler's List (1993),Drama|War
4,1221,27201,4.276,4.2495,"Godfather: Part II, The (1974)",Crime|Drama
5,904,17329,4.272,4.2311,Rear Window (1954),Mystery|Thriller
6,912,24175,4.259,4.2301,Casablanca (1942),Drama|Romance
7,1193,29702,4.248,4.2248,One Flew Over the Cuckoo's Nest (1975),Drama
8,750,23082,4.248,4.2176,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War
9,2019,11486,4.276,4.2155,Seven Samurai (Shichinin no samurai) (1954),Action|Adventure|Drama


In [7]:
def evaluate_recall_at_10(movie_ids, model_name):
    ids_tuple = tuple(movie_ids)
    query = f"""
    SELECT
        COUNT(*) AS relevant_test_ratings,
        COUNT(*) FILTER (WHERE movieId IN {ids_tuple}) AS hits
    FROM read_parquet('{TEST_PATH.as_posix()}')
    WHERE rating >= 4.0
    """
    res = con.execute(query).df()
    res["recall_at_10"] = res["hits"] / res["relevant_test_ratings"]
    res.insert(0, "model", model_name)
    return res

eval_simple = evaluate_recall_at_10(top_10["movieId"].tolist(), "Simple Popularity")
eval_weighted = evaluate_recall_at_10(top_10_weighted["movieId"].tolist(), "Weighted Popularity")

comparison = pd.concat([eval_simple, eval_weighted], ignore_index=True)
comparison

,model,relevant_test_ratings,hits,recall_at_10
0,Simple Popularity,80321,3266,0.040662
1,Weighted Popularity,80321,2252,0.028037
